In [46]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
df = pd.read_csv("./fmnist_small.csv")

X = df.loc[:,'pixel1':]

y = df.loc[:,'label']

X = X.to_numpy()
y = y.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

X_train = X_train.reshape(-1,1,28,28)
X_test = X_test.reshape(-1,1,28,28)

y_train = encoder.fit_transform(y_train.reshape(-1,1))
y_test = encoder.transform(y_test.reshape(-1,1))


In [51]:
class Layer:
    def __init__(self,n_neurons, n_inputs):
        self.weights = 0.01*np.random.randn(n_inputs,n_neurons)
        self.biases = np.zeros(1,n_neurons)
    def forward(self, x):
        self.output = np.dot(self.inputs,self.weights) + self.biases
        self.inputs = self.inputs
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T , dvalues)
        self.dbiases = np.sum(dvalues,axis = 0,keepdims = True)
        self.dinputs = np.dot(dvalues , self.weights.T)

class Activation_ReLU:
    def __init__(self):
        pass
    def forward(self,inputs):
        self.output = np.maximum(0,inputs)
        self.inputs = inputs
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs < 0 ] = 0

class Activation_Softmax:
    def __init__(self):
        pass
    def forward(self, inputs):
        self.inputs = inputs
        self.outputs = np.exp(np.clip((self.inputs - np.max(self.inputs, axis =1, keepdims= True)), 1e-7, 1-1e-7))
        self.outputs = self.outputs / np.sum(self.outputs, axis = 1)

class LossAndSoftmax:
    def __init__(self):
        pass
    def forward(self, y_pred, y):
        y_pred_clipped = np.clip(y_pred, 1e-7, 1-1e-7)
        correct_confidence = np.sum(y_pred * y,axis = 1)
        neg_loss = -np.log(correct_confidence)
        return neg_loss
    def backward(self,dvalues, y_true):
        self.dinputs = dvalues.copy()
        samples = len(dvalues)
        self.dinputs[range(samples), y_true] -=1
        self.dinputs /= samples

In [ ]:
class Conv2D:
    def __init__(self, filters, kernel_size=3, stride = 1, padding = 0, in_channels = 1):
        self.filters = filters
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        fan_in = kernel_size * kernel_size * in_channels
        self.weights = np.random.randn(kernel_size, kernel_size,in_channels, filters) * np.sqrt(2.0/fan_in)
        self.biases = np.zeros(filters)
        self.inputs = None
        self.padded_inputs = None

    def im2col(self, inputs):
        batch_size, in_h, in_w, in_c = inputs.shape
        if self.padding > 0:
            inputs = np.pad(inputs, ((0,0),(self.padding,self.padding), (self.padding,self.padding), (0,0)),mode = 'constant')
            in_h += 2*self.padding
            in_w += 2*self.padding
        self.padded_inputs = inputs
        out_h = (in_h - self.kernel_size) // self.stride + 1
        out_w = (in_w - self.kernel_size) // self.stride + 1
        col = np.zeros((batch_size * out_h * out_w, self.kernel_size * self.kernel_size * in_c))

        for y in range(self.kernel_size):
            y_max = y+self.stride * out_h
            for x in range(self.kernel_size):
                x_max = x+ self.stride * out_w
                for c in range(in_c):
                    col[:,(y*self.kernel_size + x) * in_c + c] = inputs[:,y:y_max:self.stride,x:x_max:self.stride, c].ravel()
        return col, out_h , out_w

    def forward(self,inputs):
        self.inputs = inputs
        batch_size, in_h, in_w, in_c = inputs.shape
        if self.weights.shape[2] != in_c:
            fan_in = self.kernel_size * self.kernel_size * in_c
            self.weights = np.random.randn(self.kernel_size, self.kernel_size, in_c, self.filters) * np.sqrt(2.0/ fan_in)
        col, out_h, out_w = self.im2col(inputs)
        W_col = self.weights.reshape(-1,self.filters)
        output = np.dot(col, W_col) + self.biases
        self.output = output.reshape(batch_size, out_h, out_w, self.filters)
        return self.output

    def col2im(self,col, input_shape):
        batch_size, in_h, in_w, in_c = input_shape

        if self.padding > 0:
            in_h += 2*self.padding
            in_w += 2*self.padding

        out_h = (in_h - self.kernel_size) // self.stride + 1
        out_w = (in_w - self.kernel_size) // self.stride + 1

        img = np.zeros((batch_size, in_h, in_w, in_c))
        for y in range(self.kernel_size):
            y_max = y+self.stride * out_h
            for x in range(self.kernel_size):
                x_max = x+ self.stride * out_w
                for c in range(in_c):
                   img[:, y:y_max:self.stride, x:x_max:self.stride, c] +=  col[:,(y*self.kernel_size+x)*in_c +c].reshape(batch_size, out_h, out_w)
        return img

    def backward(self,dvalues):
        batch_size, out_h, out_w , out_c = dvalues.shape
        dvalues_flat = dvalues.reshape(-1,self.filters)
        col, _, _ = self.im2col(self.inputs)
        self.dweights = np.dot(col.T, dvalues.flat).reshape(self.weights.shape)
        self.dbiases = np.sum(dvalues_flat, axis = 0)

        W_col = self.weights.reshape(-1,self.filters)
        dcol = np.dot(dvalues_flat, W_col.T)
        dinputs_padded = self.col2im(dcol, self.inputs.shape)

        if self.padding >0:
            self.dinputs = dinputs_padded[:,self.padding: -self.padding, self.ppadding:-self.padding, :]
        else:
            self.dinputs = dinputs_padded
        return self.dinputs


class MaxPool2D:
    def __init__(self, pool_size = 2, stride = 2):
        self.pool_size = pool_size
        self.stride = stride
        self.inputs = None
        self.mask = None
        self.output = None

    def forward(self, inputs):
        self.inputs = inputs
        batch_size, in_h, in_w, in_c = inputs.shape
        out_h = (in_h - self.pool_size) // self.stride + 1
        out_w = (in_w - self.pool_size) //self.stride + 1

        reshaped = inputs[:, :out_h*self.stride, :out_w *self.stride, :].copy()
        reshaped = reshaped.reshape(batch_size, out_h, self.stride, out_w, self.stride, in_c)
        reshaped = reshaped.transpose(0,1,3,5,2,4)
        reshaped = reshaped.reshape(batch_size, out_h, out_w, in_c, -1)

        self.output = np.max(reshaped, axis = -1)
        max_indices = np.argmax(reshaped, axis = -1)

        self.mask = np.zeros_like(reshaped, dtype = bool)
        batch_idx, h_idx, w_idx, c_idx = np.indices(max_indices.shape)
        self.mask[batch_size, h_idx, w_idx, c_idx, max_indices] = True

        return self.output

    def backward(self, dvalues):
        batch_size, out_h, out_w, in_c = dvalues.shape
        expanded_grad = np.zeros_like(self.mask, dtype = np.float32)
        expanded_grad[self.mask] = dvalues.ravel()

        expanded_grad = expanded_grad.reshape(batch_size, out_h, out_w, in_c, self.stride, self.output)
        expnaded_grad = expanded_grad.transpose(0,1,4,2,5,3)
        self.dinputs = expanded_grad.reshape(batch_size, out_h*self.stride, out_w * self.stride, in_c)

        if self.dinputs.shape != self.inputs.shape:
            pad_h = self.inpus.shape[1] - self.dinputs.shape[1]
            pad_w = self.inputs.shape[2] - self.dinputs.shape[2]
            self.dinputs = np.pad(self.dinputs, ((0,0), (0,pad_h), (0,pad_w), (0,0)))

        return self.dinputs

In [ ]:
class Optimizer_SGD:
    def __init__(self, learning_rate = 1.0, decay = 0., momentum = 0.):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.momentum = momentum
        self.iterations = 0
    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.current_learning_rate / (1+(self.decay * self.iterations))
    def update_params(self,layer):
        if self.momentum:
            if not hasattr(layer, 'weight_momentums'):
                layer.weight_momentums = np.zeros_like(layer.weights)
                layer.bias_momentums = np.zeros_like(layer.biases)
            weight_updates = self.momentum * layer.weight_momentums - self.learning_rate * layer.dweights
            layer.weight_momentums = weight_updates
            bias_updates = self.momentum * layer.bias_momentums - self.learning_rate * layer.dbiases
            layer.bias_momentums = bias_updates

        else:
            weight_updates = - self.learning_rate * layer.dweights
            bias_updates = -self.current_learning_rate * layer.dbiases
        layer.weights += weight_updates
        layer.biases += bias_updates

    def post_update_params(self):
        self.iterations += 1

In [ ]:
conv1 = Conv2D(filters=32, kernel_size= 3, padding= 1)
activation1 = Activation_ReLU()
pool1 = MaxPool2D(pool_size= 2, stride= 2)
flatten = lambda x: x.reshape(x.shape[0], -1)
layer1 = Layer(14*14*32, 128)
activation2 = Activation_ReLU()
layer2 = Layer(128,10)
softmax2 = Activation_Softmax()
loss_func = LossAndSoftmax()
optimizer = Optimizer_SGD(learning_rate=0.1, decay= 1e-3, momentum= 0.9)

In [ ]:
def forward_pass(X_batch, y_true):
    conv1.forward(X_batch)
    activation1.forward(conv1.output)
    pool1.forward(activation1.output)

    flatten_output = pool1.output.reshape(X_batch.shape[0], -1)
    layer1.forward(flatten_output)
    activation2.forward(layer1.output)
    layer2.forward(activation2.output)
    loss = loss_func.forward(layer2.output, y_true)
    return loss

In [ ]:

def backward_pass(dvalues, y_batch):
    loss_func.backward(dvalues, y_batch)
    layer2.backward(loss_func.dinputs)
    activation2.backward(layer2.dinputs)
    layer1.backward(activation2.dinputs)

    pool_grad = layer1.dinputs.reshape(
        pool1.output.shape[0],  # batch
        pool1.output.shape[1],  # height
        pool1.output.shape[2],  # width
        pool1.output.shape[3]   # channels
    )
    pool1.backward(pool_grad)
    activation1.backward(pool1.dinputs)
    conv1.backward(activation1.dinputs)